# Automated Codebook

## 1. Imports

In [1]:
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 2. Configuration

In [2]:
MODEL_NAME = "gpt-oss:120b"
TEMPERATURE = 0.3 # taken from Nyaaba et al (2026)
NUM_CTX = 32768  # Context used for prediction. Could push higher

NUM_INTERVIEWS = 5

# Chunking Settings
CHUNK_SIZE = 4000  # Roughly 800-1000 words per chunk to prevent "Lost in the Middle"
CHUNK_OVERLAP = 400  # Overlap across chunks

## 3. Prompt Templates

### 3.1. System prompt: A variation on Tree of Thought (ToT)

Five experts that analyse each chunk slightly differently with expert 5 coordinating.  Differs from standard ToT as no experts leave the room.

In [3]:
EXPERTS_SETUP = """
Five different experts are collaboratively identifying themes across interviews regarding the reuse of a
simulation model of a stroke pathway.

- Expert 1 focuses on barriers and challenges in reuse.
- Expert 2 focuses on what went well in using the model (facilitators, enablers, successes).
- Expert 3 focuses on what needs to be improved to enable reuse
- Expert 4 focuses on anything unexpected, surprising, contradictory, or outlier observations.
- Expert 5 acts as the Lead Synthesizer and Editor.

Expert 5 has a strict responsibility to:
1. aggressively merge overlapping or near-duplicate codes;
2. prevent codebook bloat;
3. preserve traceability across ALL interviews i.e. do not tracibility to early interviews;
4. keep only the most representative supporting evidence for each code;
5. ensure that codes remain distinct, meaningful, and reusable across interviews.
"""

### 3.2 Codebook prompt

The prompt will vary depending on the interview number. For interview 2 onwards the current codebook is appended. Implemented using `jinja2` if else. 

In [4]:
codebook_template = """
{{ experts_setup }}

{% if current_codebook %}
Task: 
Update the existing codebook below using the new transcript chunk. 

CURRENT CODEBOOK:
{{ current_codebook }}

Instructions:
1. Apply Existing: If new evidence fits an existing code, add the quote. 
2. Create New: Only create a new code if the idea cannot be absorbed into an existing one.
3. Consolidate (Expert 5): Aggressively merge overlapping codes. Delete weak or highly specific micro-codes. Keep the codebook compact.

Evidence Rules:
- Keep a MAXIMUM of 3 representative quotes per code. Replace weaker quotes with better ones.
- Ensure diversity of quotes across interviewees.
- Quotes must be exact and include the ID in brackets: [{{ interview_id }}] "exact quote"

Output:
Return ONLY the updated Markdown table with columns:  Expert Role | Emerging Code | Definition | Relevant Interviews |Source Evidence.

{% else %}
Task: 
Extract emerging codes from the transcript chunk below to create an initial codebook. Do not create a code for each observation.

Instructions:
1. Experts 1-4 independently identify relevant codes based on their assigned perspectives.
2. Refine (Expert 5): Review all candidate codes. Merge overlaps, remove redundancies, and tighten definitions to ensure codes are distinct and broadly applicable.

Evidence Rules:
- Keep a MAXIMUM of 3 representative quotes per code.
- Quotes must be exact and include the ID in brackets: [{{ interview_id }}] "exact quote"

Output:
Return ONLY a Markdown table with columns: Expert Role | Emerging Code | Definition | Relevant Interviews |Source Evidence.

{% endif %}

Transcript Chunk ({{ interview_id }}):
{{ transcript_chunk }}
"""

### 3.3 Setup prompt template

In [5]:
prompt = PromptTemplate.from_template(
    codebook_template, 
    template_format="jinja2"
)

## 4. Initialise Ollama and LLM

In [6]:
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    num_ctx=NUM_CTX
)

## 5. Setup chain and chunkifier

In [7]:
# Define the Chain (LCEL)

# chain = filled in prompt template -> chosen LLM -> cleaned up response
analysis_chain = prompt | llm | StrOutputParser()

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

## 6. Run analysis function

In [9]:
def run_analysis():
    current_codebook = ""
    
    for i in range(1, NUM_INTERVIEWS + 1):
        filename = f"../data/interview_{i}.txt"
        
        if not os.path.exists(filename):
            print(f"File {filename} not found. Skipping...")
            continue
            
        with open(filename, 'r', encoding='utf-8') as f:
            interview_text = f.read()
            
        # Format a clean ID for "qualitative traceability"
        current_interview_id = filename.replace(".txt", "").replace("_", " ").title()
            
        # Split the interview into chunks to help LLM avoid "missing the middle"
        chunks = text_splitter.split_text(interview_text)
        print(f"Divided {filename} into {len(chunks)} chunks.")
        
        for chunk_idx, chunk_text in enumerate(chunks, 1):
            print(f"  -> Processing {current_interview_id} - Chunk {chunk_idx}/{len(chunks)}...")
            
            # LangChain pipeline with the mapped variables
            current_codebook = analysis_chain.invoke({
                "experts_setup": EXPERTS_SETUP,
                "current_codebook": current_codebook,
                "transcript_chunk": chunk_text,
                "interview_id": current_interview_id
            })
            
            # Save progress after EVERY chunk to prevent data loss on long runs
            output_filename = f"../output/codebook_after_interview_{i}_chunk_{chunk_idx}.md"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(current_codebook)
                
        print(f"✅ Finished all chunks for {filename}.")
        
    print(f"\n ✅ Codebook complete")

In [10]:
run_analysis()

Divided ../data/interview_1.txt into 15 chunks.
  -> Processing ../Data/Interview 1 - Chunk 1/15...
  -> Processing ../Data/Interview 1 - Chunk 2/15...
  -> Processing ../Data/Interview 1 - Chunk 3/15...
  -> Processing ../Data/Interview 1 - Chunk 4/15...
  -> Processing ../Data/Interview 1 - Chunk 5/15...
  -> Processing ../Data/Interview 1 - Chunk 6/15...
  -> Processing ../Data/Interview 1 - Chunk 7/15...
  -> Processing ../Data/Interview 1 - Chunk 8/15...
  -> Processing ../Data/Interview 1 - Chunk 9/15...
  -> Processing ../Data/Interview 1 - Chunk 10/15...
  -> Processing ../Data/Interview 1 - Chunk 11/15...
  -> Processing ../Data/Interview 1 - Chunk 12/15...
  -> Processing ../Data/Interview 1 - Chunk 13/15...
  -> Processing ../Data/Interview 1 - Chunk 14/15...
  -> Processing ../Data/Interview 1 - Chunk 15/15...
✅ Finished all chunks for ../data/interview_1.txt.
Divided ../data/interview_2.txt into 13 chunks.
  -> Processing ../Data/Interview 2 - Chunk 1/13...
  -> Processing